# SchemaSmith LoRA SFT on Tiny Aya

Free T4 notebook for Colab or Kaggle. It fine-tunes `CohereLabs/tiny-aya-global` with AlignTune's TRL backend and LoRA.

This file does not record a training result. Run it yourself on a GPU. If CUDA is missing, the training cell skips.

Before the first GPU run:

1. Open the repository root as the working directory.
2. Select a free T4 runtime.
3. Accept the model license on the Hugging Face model card for `CohereLabs/tiny-aya-global`.
4. Provide `HF_TOKEN` in the environment, or as a Colab secret of that name. The token is not stored in this repository.

The install below pulls AlignTune, which depends on CuratorKIT. Those packages stay outside the MIT dependency file. See `requirements-lexsi.txt` and `docs/lexsi_apis.md`.


## Install

Runtime dependencies for training only. Re-run after a runtime restart.

In [ ]:
import subprocess
import sys
from pathlib import Path

if not (Path.cwd() / "pyproject.toml").exists():
    raise SystemExit("Start the notebook from the repository root.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-lexsi.txt"]
)
print("install finished")


## Hugging Face login

Tiny Aya is gated. A missing token does not stop the data cells. It does stop the weight download.

In [ ]:
import os

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
try:
    from google.colab import userdata

    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import login

    login(token=token, add_to_git_credential=False)
    print("HF login ok")
else:
    print("No HF_TOKEN. Gated weights will not download.")


## Data

24 synthetic tickets, seed 42. The local contract gate writes `data/notebook/local`. CuratorKIT then cleans, deduplicates, and splits the same rows into `data/notebook/curatorkit/curated`. Counts printed here come from that run.


In [ ]:
from pathlib import Path

from schemasmith.curation.curatorkit import curate_with_curatorkit
from schemasmith.curation.local import curate_local
from schemasmith.data.generator import generate_tickets
from schemasmith.data.loader import write_jsonl

out = Path("data/notebook")
records = generate_tickets(24, seed=42)
write_jsonl(out / "raw.jsonl", [record.to_dict() for record in records])

local = curate_local(records, out / "local", seed=42)
print("local", local.summary()["split_counts"])

curator = curate_with_curatorkit(records, out / "curatorkit", seed=42)
print("curatorkit passed", curator["passed_count"], "rejected", curator["rejected_count"])
print(curator["stage_counts"])


## One training row

In [ ]:
import json

sample = next((out / "curatorkit" / "curated").rglob("sft_alpaca.jsonl"))
row = json.loads(sample.read_text(encoding="utf-8").splitlines()[0])
print(sample)
print(row["input"][:400])
print(row["output"][:400])


## Train

Uses `schemasmith.training.lora_sft_kwargs`: TRL backend, LoRA, 4-bit load, subset `sft_alpaca`, 40 steps. No weights are pushed.


In [ ]:
from pathlib import Path

import torch
from aligntune.core.backend_factory import create_sft_trainer

from schemasmith.training import lora_sft_kwargs

dataset_dir = Path("data/notebook/curatorkit/curated")
if not dataset_dir.is_dir():
    raise SystemExit("Run the data cell first.")

if not torch.cuda.is_available():
    print("No GPU. Training skipped. Select a free T4 and rerun this cell.")
else:
    kwargs = lora_sft_kwargs(
        dataset_dir=str(dataset_dir.resolve()),
        output_dir="output/lora-tiny-aya",
    )
    print(kwargs["model_name"], kwargs["backend"], kwargs["subset"], "peft", kwargs["use_peft"])
    trainer = create_sft_trainer(**kwargs)
    trainer.train()
    print("train() returned")


## Pipeline check

This cell scores gold JSON with the contract validator so the export can be read without a model. It is not a base-model result and not a fine-tune result.


In [ ]:
import json
from pathlib import Path

from schemasmith.data.loader import load_alpaca_jsonl
from schemasmith.eval.harness import evaluate_records

test_file = Path("data/notebook/curatorkit/curated/test/sft_alpaca.jsonl")
held_out = load_alpaca_jsonl(test_file)
card = evaluate_records(
    held_out,
    generate=lambda _texts: [json.dumps(record.gold) for record in held_out],
)
print("pipeline check on gold JSON, not a model measurement")
print("n", card.n, "schema_valid_rate", card.schema_valid_rate)


## Base-model eval, not run here

When you have a GPU and the gated weights, AuditKIT 1.1.6 accepts this shape. It was checked with a callable model, not with Tiny Aya:

```python
import auditkit as ak
from schemasmith.paths import schema_path

result = ak.evaluate(
    samples,
    "hf:CohereLabs/tiny-aya-global",
    scorers=[
        ak.IsJson(),
        ak.JsonSchemaValid(schema_path(), backend="jsonschema", strip_fences=True),
        ak.FieldF1(normalize_numbers=True, number_tol=1e-6, strip_fences=True),
    ],
)
```

`IsJson` does not strip markdown fences. The contract validator does. Keep both numbers. Write the strings to a predictions JSONL and run `schemasmith eval` so the contract rates are in the same report. Leave the results table in the README as TBD until that run exists.
